# Trajectories of magnetic soft robot

### Imports

In [1]:
import numpy as np
import jax.numpy as jnp
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px

from softmobility import SoftBody, FlowBodySolver, Taylor_Green_flow, gravity_field, constant_scalar

np.set_printoptions(precision=3, linewidth=100, suppress=True, sign=" ")

## Simulation of default parameters

### YAML file

In [2]:
yaml_data = """
# Variable Prefixes (Optional)
dof_names:
  - phi
design_names:      
  - spring_k
  - mass    
  - radius
input_names:
  - gravity
  - active_force
    
# Default Values (Optional)
defaults:
  spring_k: .5           
  radius: .5
  mass: 5
  _spr_: 20
  _rad_: 1
  
# Spheres (Mandatory)
spheres:
  - # Sphere 0 #################
    radius: 1
    position: [0, 0, 1]                    
    orientation: [phi, 0, 0]
    force: [-mass*gravity0, -mass*gravity1, -mass * gravity2]
    torque: [-_spr_ * spring_k * phi, 0, 0]              

  - # Sphere 1 #################
    radius: _rad_ * radius               
    position: [0, 0, -radius]
    orientation: [-phi/_rad_/radius, 0, 0]
    force: [mass*gravity0, mass*gravity1 + active_force * sin(phi/_rad_/radius), mass * gravity2 + active_force * cos(phi/_rad_/radius)]
    torque: [_spr_ * spring_k * phi, 0, 0]
"""

In [3]:
yaml_data_rigid = """
# Variable Prefixes (Optional)
design_names:      
  - mass    
  - radius
input_names:
  - gravity
  - active_force
    
# Default Values (Optional)
defaults:
  radius: .5
  mass: 5
  _rad_: 1
  
# Spheres (Mandatory)
spheres:
  - # Sphere 0 #################
    radius: 1
    position: [0, 0, 1]                    
    orientation: [0, 0, 0]
    force: [-mass*gravity0, -mass*gravity1, -mass * gravity2]
    torque: [0, 0, 0]              

  - # Sphere 1 #################
    radius: _rad_ * radius               
    position: [0, 0, -radius]
    orientation: [0, 0, 0]
    force: [mass*gravity0, mass*gravity1, mass * gravity2 + active_force]
    torque: [0, 0, 0]
"""

### Soft Body

In [4]:
mybody= SoftBody(yaml_data, verbose=False)
myrigidbody = SoftBody(yaml_data_rigid, verbose=False)

### Parameters of flow and gravity field

In [7]:
# Calculating force to have a swimming velocity of 1
tensors = myrigidbody.compute_fast_mobility(dofs=jnp.zeros(myrigidbody.Ndof))
FORCE_INTENSITY=1/tensors.M_H[2,-1]

In [8]:
# Create a TGV flow with with max vorticy = 1
FLOW = Taylor_Green_flow(omega=2)
# Create a gravity field 
GRAVITY = gravity_field(g=10.0)
# Create an active force scalar
FORCE = constant_scalar(value=FORCE_INTENSITY)

In [9]:
# parameters of time integration
final_time = 4 * 2 * jnp.pi  
DT = 0.2
N_TRAJECTORIES = 7

### Brute-force optimization of stiffness

In [10]:
spring_ks = np.linspace(0, 1, 11)  # [0.0, 0.1, ..., 1.0]
x_inits = np.linspace(np.pi/N_TRAJECTORIES/2, 2*np.pi, N_TRAJECTORIES, endpoint=False)
results = []

for spring_k in spring_ks:
    mybody.set_design_defaults(new_dict={'spring_k': spring_k})

    trajectories = []
    for x_init in x_inits:
        solver = FlowBodySolver(
            soft_body=mybody,
            flow=FLOW,
            input_map={"gravity": GRAVITY, "active_force": FORCE},
            dt=DT,
            init_position=[np.pi/2, x_init, 0],
            init_orientation=[0, 0, 0],
            integrator="RK2")
        trajectories.append(solver.simulate(final_time=final_time))

    end_z_positions = [float(jnp.array([t[0] for t in traj])[-1, 2]) for traj in trajectories]
    mean_velocity = np.mean(end_z_positions) / final_time
    results.append((spring_k, mean_velocity))
    print(f"spring_k={spring_k:.2f}  →  mean velocity={mean_velocity:.4f}")

# --- Summary ---
best_k, best_v = max(results, key=lambda x: x[1])
print(f"\nBest spring_k={best_k:.2f}  →  mean velocity={best_v:.4f}")

OLD default param values: ['mass', 'radius', 'spring_k'] [ 5.   0.5  0.5]
NEW default param values: ['mass', 'radius', 'spring_k'] [ 5.   0.5  0. ]
Time: 0.000 / 25.133  Integrator RK2
Time: 20.000 / 25.133  Integrator RK2
Time: 0.000 / 25.133  Integrator RK2
Time: 20.000 / 25.133  Integrator RK2
Time: 0.000 / 25.133  Integrator RK2
Time: 20.000 / 25.133  Integrator RK2
Time: 0.000 / 25.133  Integrator RK2
Time: 20.000 / 25.133  Integrator RK2
Time: 0.000 / 25.133  Integrator RK2
Time: 20.000 / 25.133  Integrator RK2
Time: 0.000 / 25.133  Integrator RK2
Time: 20.000 / 25.133  Integrator RK2
Time: 0.000 / 25.133  Integrator RK2
Time: 20.000 / 25.133  Integrator RK2
spring_k=0.00  →  mean velocity=-0.1994
OLD default param values: ['mass', 'radius', 'spring_k'] [ 5.   0.5  0. ]
NEW default param values: ['mass', 'radius', 'spring_k'] [ 5.   0.5  0.1]
Time: 0.000 / 25.133  Integrator RK2
Time: 20.000 / 25.133  Integrator RK2
Time: 0.000 / 25.133  Integrator RK2
Time: 20.000 / 25.133  Inte

### Simulations for plots

In [11]:
mybody.set_design_defaults(new_dict={
        'spring_k': best_k, 
})

trajectories = []

for x_init in x_inits:
    print("New initial position", x_init)
    solver = FlowBodySolver(
        soft_body=mybody, 
        flow=FLOW, 
        input_map={"gravity": GRAVITY, "active_force": FORCE}, 
        dt=DT, 
        init_position=[np.pi/2, x_init, 0],
        init_orientation=[0, 0, 0], 
        integrator="RK2")
    trajectories.append(solver.simulate(final_time=final_time))
    
end_z_positions = [float(jnp.array([t[0] for t in traj])[-1, 2]) for traj in trajectories]
mean_end_z = np.mean(end_z_positions)

print(f"End Z positions: {[f'{z/np.pi:.3g}π' for z in end_z_positions]}")
print(f"Mean velocity: {mean_end_z/final_time:.3g}")

OLD default param values: ['mass', 'radius', 'spring_k'] [ 5.   0.5  1. ]
NEW default param values: ['mass', 'radius', 'spring_k'] [ 5.   0.5  0.5]
New initial position 0.2243994752564138
Time: 0.000 / 25.133  Integrator RK2
Time: 20.000 / 25.133  Integrator RK2
New initial position 1.0899403083882957
Time: 0.000 / 25.133  Integrator RK2
Time: 20.000 / 25.133  Integrator RK2
New initial position 1.9554811415201774
Time: 0.000 / 25.133  Integrator RK2
Time: 20.000 / 25.133  Integrator RK2
New initial position 2.821021974652059
Time: 0.000 / 25.133  Integrator RK2
Time: 20.000 / 25.133  Integrator RK2
New initial position 3.686562807783941
Time: 0.000 / 25.133  Integrator RK2
Time: 20.000 / 25.133  Integrator RK2
New initial position 4.552103640915822
Time: 0.000 / 25.133  Integrator RK2
Time: 20.000 / 25.133  Integrator RK2
New initial position 5.417644474047704
Time: 0.000 / 25.133  Integrator RK2
Time: 20.000 / 25.133  Integrator RK2
End Z positions: ['0.257π', '10.7π', '11.1π', '10.8

In [12]:
rigid_trajectories = []

for x_init in x_inits:
    print("New initial position", x_init)
    solver = FlowBodySolver(
        soft_body=myrigidbody, 
        flow=FLOW, 
        input_map={"gravity": GRAVITY, "active_force": FORCE}, 
        dt=DT, 
        init_position=[np.pi/2, x_init, 0],
        init_orientation=[0, 0, 0], 
        integrator="RK2")
    rigid_trajectories.append(solver.simulate(final_time=final_time))
    
end_z_positions = [float(jnp.array([t[0] for t in traj])[-1, 2]) for traj in rigid_trajectories]
mean_end_z = np.mean(end_z_positions)

print(f"End Z positions: {[f'{z/np.pi:.3g}π' for z in end_z_positions]}")
print(f"Mean velocity: {mean_end_z/final_time:.3g}")

New initial position 0.2243994752564138
Time: 0.000 / 25.133  Integrator RK2
Time: 20.000 / 25.133  Integrator RK2
New initial position 1.0899403083882957
Time: 0.000 / 25.133  Integrator RK2
Time: 20.000 / 25.133  Integrator RK2
New initial position 1.9554811415201774
Time: 0.000 / 25.133  Integrator RK2
Time: 20.000 / 25.133  Integrator RK2
New initial position 2.821021974652059
Time: 0.000 / 25.133  Integrator RK2
Time: 20.000 / 25.133  Integrator RK2
New initial position 3.686562807783941
Time: 0.000 / 25.133  Integrator RK2
Time: 20.000 / 25.133  Integrator RK2
New initial position 4.552103640915822
Time: 0.000 / 25.133  Integrator RK2
Time: 20.000 / 25.133  Integrator RK2
New initial position 5.417644474047704
Time: 0.000 / 25.133  Integrator RK2
Time: 20.000 / 25.133  Integrator RK2
End Z positions: ['0.172π', '2.1π', '4.16π', '3.11π', '5.25π', '2.24π', '2.15π']
Mean velocity: 0.342


### Figure of trajectory

In [ ]:
# One color per trajectory
colors = px.colors.qualitative.Plotly[:len(trajectories)]

fig = make_subplots(
    rows=3, cols=1,
    subplot_titles=("DOF", "Position (X, Y, Z)", "Orientation (X, Y, Z)"),
    shared_xaxes=True,
    vertical_spacing=0.1
)

for i, (trajectory, x_init) in enumerate(zip(trajectories, x_inits)):
    label     = f"y₀={x_init/np.pi:.2g}π"
    color     = colors[i]
    group     = f"traj_{i}"

    positions    = jnp.array([t[0] for t in trajectory])
    orientations = jnp.array([t[1] for t in trajectory])
    dofs         = jnp.array([t[2] for t in trajectory])
    time         = jnp.linspace(0, final_time, len(trajectory))

    # Helper: only the first trace of each group appears in the legend
    def trace(y, name, row, first=False, dash='solid'):
        return go.Scatter(
            x=time, y=y,
            mode='lines',
            line=dict(color=color, width=1.5, dash=dash),
            name=label,
            legendgroup=group,
            legendgrouptitle_text=label if (first and i == 0) else None,
            showlegend=first,   # only the group's first trace gets a legend entry
            hovertemplate=f"{name}<br>t=%{{x:.2f}}<br>val=%{{y:.3f}}<extra>{label}</extra>",
        )

    fig.add_trace(trace(dofs,               "DOF",           row=1, first=True),           row=1, col=1)
    fig.add_trace(trace(positions[:, 0],    "Position X",    row=2, dash='solid'),          row=2, col=1)
    fig.add_trace(trace(positions[:, 1],    "Position Y",    row=2, dash='dash'),           row=2, col=1)
    fig.add_trace(trace(positions[:, 2],    "Position Z",    row=2, dash='dot'),            row=2, col=1)
    fig.add_trace(trace(orientations[:, 0], "Orientation X", row=3, dash='solid'),          row=3, col=1)
    fig.add_trace(trace(orientations[:, 1], "Orientation Y", row=3, dash='dash'),           row=3, col=1)
    fig.add_trace(trace(orientations[:, 2], "Orientation Z", row=3, dash='dot'),            row=3, col=1)

fig.update_layout(
    title="Trajectory Data Over Time",
    template="plotly_white",
    plot_bgcolor='white',
    paper_bgcolor='white',
    showlegend=True,
    height=400, width=600,
    legend=dict(groupclick="toggleitem"),   # click legend item → toggle one trace
                                            # double-click         → toggle whole group
)
fig.show()

In [ ]:
# --- Helper to build tick labels like "−2π", "−π/2", "0", "π", etc. ---
def pi_ticks(lo, hi, step_pi=1):
    """Return (tickvals, ticktext) with spacing of step_pi * π."""
    import math
    n_lo = math.floor(lo / (step_pi * np.pi))
    n_hi = math.ceil(hi  / (step_pi * np.pi))
    vals, texts = [], []
    for n in range(n_lo, n_hi + 1):
        v = n * step_pi * np.pi
        vals.append(v)
        k = n * step_pi          # multiple of π
        if   k == 0:  texts.append('0')
        elif k == 1:  texts.append('π')
        elif k == -1: texts.append('−π')
        else:         texts.append(f'{k}π')
    return vals, texts

def bounds_2pi(lo, hi, step_pi=2):
    """Round lo down and hi up to the nearest multiple of step_pi*π."""
    unit = step_pi * np.pi
    return np.floor(lo / unit) * unit, np.ceil(hi / unit) * unit


In [ ]:

colors = px.colors.qualitative.Plotly[:len(trajectories)]

# --- Compute global bounds across ALL trajectories ---
all_y = np.concatenate([np.array(jnp.array([t[0] for t in traj])[:, 1]) for traj in trajectories])
all_z = np.concatenate([np.array(jnp.array([t[0] for t in traj])[:, 2]) for traj in trajectories])

Y_MIN, Y_MAX = bounds_2pi(all_y.min(), all_y.max())
Z_MIN, Z_MAX = bounds_2pi(all_z.min(), all_z.max())

GRID_STEP  = 1
y_ticks, y_labels = pi_ticks(Y_MIN, Y_MAX, step_pi=GRID_STEP)
z_ticks, z_labels = pi_ticks(Z_MIN, Z_MAX, step_pi=GRID_STEP)

ASPECT     = (Y_MAX - Y_MIN)
FIG_WIDTH  = 300
FIG_HEIGHT = int(FIG_WIDTH * (Z_MAX - Z_MIN) / ASPECT)

fig = go.Figure()

# --- One trace per trajectory ---
for i, (trajectory, x_init) in enumerate(zip(trajectories, x_inits)):
    positions = jnp.array([t[0] for t in trajectory])
    y_pos = np.array(positions[:, 1])
    z_pos = np.array(positions[:, 2])
    label = f"y₀={x_init/np.pi:.2g}π"

    fig.add_trace(go.Scatter(
        x=y_pos, y=z_pos,
        mode='lines',
        line=dict(color=colors[i], width=2),
        name=label,
        hovertemplate=f"Y=%{{x:.3f}}<br>Z=%{{y:.3f}}<extra>{label}</extra>",
    ))
    # Start / end markers
    fig.add_trace(go.Scatter(
        x=[y_pos[0], y_pos[-1]], y=[z_pos[0], z_pos[-1]],
        mode='markers',
        marker=dict(color=colors[i], size=8, symbol=['circle', 'square']),
        legendgroup=label, showlegend=False,
        hovertemplate=f"%{{text}}<extra>{label}</extra>",
        text=['start', 'end'],
    ))

fig.update_layout(
    template='plotly_white',
    plot_bgcolor='white',
    paper_bgcolor='white',
    width=FIG_WIDTH,
    height=FIG_HEIGHT,
    margin=dict(l=20, r=20, t=20, b=20),
    showlegend=False,
    xaxis=dict(
        range=[Y_MIN, Y_MAX],
        scaleanchor='y', scaleratio=1,
        constrain='domain',
        tickvals=y_ticks, showticklabels=False,
        showgrid=True, gridcolor='lightgrey', gridwidth=1,
        zeroline=True, linecolor='lightgrey', mirror=True,
        title=None,
    ),
    yaxis=dict(
        range=[Z_MIN, Z_MAX],
        constraintoward='center',
        tickvals=z_ticks, showticklabels=False,
        showgrid=True, gridcolor='lightgrey', gridwidth=1,
        zeroline=True, linecolor='lightgrey', mirror=True,
        title=None,
    ),
)

fig.show()